# OSM support-zone features


Read the pinned Arizona, Canary Islands, Chile and Bolivia extracts. Select roads/railways, named places, amenities, public-transport points and mapped building/facility areas; clip to explicit support boxes and export GeoJSON in longitude/latitude. Use these only for access/logistics, separately from terrain similarity.

Default support boxes equal the existing candidate boxes. Expand them only after checking extract coverage and operational needs; country borders can leave gaps. A disk-backed node-location index is created temporarily while each PBF is scanned (area assembly requires two passes). Allow several GB of temporary disk space. Overlap deduplication prefers the highest OSM version per original object type/ID; equal versions use the first source. Incomplete geometries are counted, and absent relations/features must not be interpreted as absence on the ground.


Run cells in order manually. Raw inputs are read-only. Each processing run writes to a new directory under `data/processed/`; only a `provenance.json` with `status: complete` indicates successful completion. See the [notebook index](../README.md) for environment setup and dependencies.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or a folder under notebooks")
sys.path.insert(0, str(ROOT / "notebooks"))
print("Project:", ROOT)


In [ ]:
import json
import hashlib
from processing import AREAS, Run, sha256
from osm_processing import extract_support

SUPPORT_BOXES = {name: list(value["bbox"]) for name, value in AREAS.items()}
EXTRACTS = {"northern_arizona": ["arizona"], "lanzarote": ["canary-islands"],
            "atacama_san_pedro": ["chile", "bolivia"]}
base = ROOT / "data/raw/earth/osm"
selection_path = base / "manifests/selection.json"
selection = json.loads(selection_path.read_text())
sources = {s["region"]: s for s in selection["sources"]}
assert set(sources) >= {r for regions in EXTRACTS.values() for r in regions}
for area, bounds in SUPPORT_BOXES.items():
    west, south, east, north = bounds
    assert -180 <= west < east <= 180 and -90 <= south < north <= 90
    print(area, bounds, EXTRACTS[area])


In [ ]:
with Run(ROOT, "osm_support", {"support_boxes": SUPPORT_BOXES, "extracts": EXTRACTS,
         "attribution": selection["attribution"], "license": selection["license"]}, "preprocessing/04_osm_support_zones.ipynb") as run:
    run.input(selection_path)
    run.input(ROOT / "notebooks/osm_processing.py")
    checked = set()
    results = {}
    for area, bounds in SUPPORT_BOXES.items():
        paths = []
        for region in EXTRACTS[area]:
            source = sources[region]
            path = base / "extracts" / source["filename"]
            receipt_path = path.with_suffix(path.suffix + ".json")
            if path not in checked:
                receipt = json.loads(receipt_path.read_text())
                assert receipt["source"] == source and path.stat().st_size == source["size_bytes"]
                assert sha256(path) == receipt["sha256"]
                digest = hashlib.md5()
                with path.open("rb") as stream:
                    for block in iter(lambda: stream.read(1024 * 1024), b""):
                        digest.update(block)
                assert digest.hexdigest() == source["md5"]
                run.input(path)
                run.input(receipt_path)
                checked.add(path)
            paths.append(path)
        results[area] = extract_support(paths, bounds, run.path(f"earth/{area}/support_features.geojson"))
        run.path(f"earth/{area}/quality.json").write_text(json.dumps(results[area], indent=2), encoding="utf-8")
        print(area, results[area])
    run.record["summaries"] = results
print("Completed:", run.directory)


## Interpretation
Retain **© OpenStreetMap contributors** attribution and ODbL terms with derived databases. Inspect diagnostic counts and mapping completeness. Access tags are evidence to investigate, not permission to use a road or site. This notebook does not calculate emergency response times, certify safety, or route across disconnected networks.
